# Exercise 2: Branch membership and transfer

*The Turkic World · version 0.1.0*

> Using the branch table, identify: (a) which branch contains Uzbek; (b) which branch Kazakh and Kyrgyz both belong to; (c) which branch Chuvash belongs to and why it is unusual. Based on genealogical proximity alone, predict which language would be a better transfer source for Kyrgyz NLP: Turkish or Kazakh? Justify your answer.

The notebook answers the questions from the branch table and then adds a small measurement: how much written overlap Kyrgyz has with Kazakh and with Turkish on parallel sentences.

**Setup.** `pip install "turkicnlp==0.5.0" requests pandas`. Cells that query websites need internet access; if a request fails, the cell records the failure instead of a number. An unavailable value is not zero.

In [ ]:
import datetime
import requests
import pandas as pd

ACCESSED = datetime.date.today().isoformat()        # record the access date with every observation
HEADERS = {"User-Agent": "turkic-nlp-book-exercises/0.1 (educational use)"}

def get_json(url, **params):
    """Return parsed JSON, or {'_error': ...} if the request fails (never a fake value)."""
    try:
        r = requests.get(url, params=params or None, headers=HEADERS, timeout=30)
        r.raise_for_status()
        return r.json()
    except Exception as exc:
        return {"_error": f"{type(exc).__name__}: {exc}"}

def get_text(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        return r.text
    except Exception as exc:
        return None

def github_org_repos(org):
    """All public repositories of a GitHub organisation (paged listing; avoids the search API's low rate limit)."""
    repos, page = [], 1
    while True:
        d = get_json(f"https://api.github.com/orgs/{org}/repos", per_page=100, page=page)
        if not isinstance(d, list):
            return repos, d.get("_error")
        repos += d
        if len(d) < 100:
            return repos, None
        page += 1

In [ ]:
BRANCH = {"Turkish": "SW (Oghuz)", "Kazakh": "NW (Kipchak)", "Kyrgyz": "NW (Kipchak)",
          "Uzbek": "SE (Karluk)", "Uyghur": "SE (Karluk)", "Chuvash": "Oghur"}
print("(a) Uzbek:", BRANCH["Uzbek"])
print("(b) Kazakh and Kyrgyz:", BRANCH["Kazakh"], "/", BRANCH["Kyrgyz"])
print("(c) Chuvash:", BRANCH["Chuvash"])

**Measuring written similarity.** The X-WMT test sets of the Turkic Interlingua project translate the same English news sentences into several Turkic languages. Aligning them through the English source gives Kyrgyz, Kazakh and Turkish versions of the same sentences. We compare character-trigram overlap (Dice) in the national scripts and after conversion to the Common Turkic Alphabet (CTS) with TurkicNLP. Overlap measures shared strings; it is evidence about the text representation, not about transfer accuracy.

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.transliterator import Transliterator

BASE = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/master/xwmt/test"

def xwmt(code):
    """English→code X-WMT test pairs as {English sentence: translation}."""
    en, xx = get_text(f"{BASE}/en-{code}/en-{code}.en"), get_text(f"{BASE}/en-{code}/en-{code}.{code}")
    if en is None or xx is None:
        raise RuntimeError(f"could not download X-WMT en-{code}")
    return dict(zip(en.splitlines(), xx.splitlines()))

side = {code: xwmt(code) for code in ["ky", "kk", "tr"]}
common = [e for e in side["ky"] if e in side["kk"] and e in side["tr"]]
print("sentences aligned through English:", len(common))

to_cts = {"ky": Transliterator("kir", Script.CYRILLIC, Script.COMMON_TURKIC),
          "kk": Transliterator("kaz", Script.CYRILLIC, Script.COMMON_TURKIC),
          "tr": Transliterator("tur", Script.LATIN, Script.COMMON_TURKIC)}

def trigrams(text):
    t = " ".join(text.lower().split())
    return {t[i:i + 3] for i in range(len(t) - 2)}

def dice(a, b):
    A, B = trigrams(a), trigrams(b)
    return 2 * len(A & B) / (len(A) + len(B)) if A and B else 0.0

def mean_dice(x, y, convert):
    scores = []
    for e in common:
        a, b = side[x][e], side[y][e]
        if convert:
            a, b = to_cts[x].transliterate(a), to_cts[y].transliterate(b)
        scores.append(dice(a, b))
    return sum(scores) / len(scores)

pd.DataFrame([{"pair": f"Kyrgyz–{name}", "native script": round(mean_dice("ky", c, False), 3),
               "CTS": round(mean_dice("ky", c, True), 3)} for c, name in [("kk", "Kazakh"), ("tr", "Turkish")]])

## Worked solution

**(a)** Uzbek belongs to the **SE (Karluk)** branch, with Uyghur.

**(b)** Kazakh and Kyrgyz both belong to the **NW (Kipchak)** branch.

**(c)** Chuvash is the only living member of the **Oghur** branch. Johanson calls it the only surviving descendant of non-Common (Oghur) Turkic, and it is the only living "r-language": it has *r* where the other Turkic languages have *z* (the classic criterion is the word for 'nine'). It is therefore the most divergent Turkic language for transfer [Johanson 2021, pp. 23, 25].

**Prediction.** On genealogy alone, **Kazakh** is the better candidate source for Kyrgyz: both are Kipchak, while Turkish is Oghuz (SW). Two further observations point the same way. Johanson notes that Kyrgyz and Kazakh have become increasingly mutually intelligible through recent convergence (p. 72), and both are written in Cyrillic.

**What the measurement adds.** On the 400 aligned X-WMT sentences (run on 18 October 2026), Kyrgyz–Kazakh trigram overlap is about 0.26 in native script and 0.30 in CTS. Kyrgyz–Turkish overlap is about 0.03 in native script, mostly because the scripts differ, and 0.18 in CTS. Script conversion narrows the gap but Kazakh stays closer. Genealogy, script and overlap all favour Kazakh here, so this case cannot tell which factor matters. Turkish still has far more training data, which is why the choice should be tested on a Kyrgyz task (Meaning and Retrieval).

## Your turn — extension

Repeat the measurement for another target, for example Kazakh against Kyrgyz and Turkish, or add Uzbek (`uz`, Latin script, `Transliterator("uzb", Script.LATIN, Script.COMMON_TURKIC)`). Does the ranking by overlap follow the branch table?